# PS- 3C: Easy LSTM Autoencoder Version

This is the LSTM comparison version. The existing PS- 3C.ipynb remains the simpler dense-autoencoder version.


In [1]:
# Import all required packages first
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras import models, layers


In [2]:
# 1. Load data, separate x and y, then keep normal rows for training.
data = pd.read_csv("energy consumption.csv")
data = data.fillna(data.mean(numeric_only=True))
x = data.drop(["Timestamp", "Anomaly_Label"], axis=1)
y = data["Anomaly_Label"]
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)
normal = x_train[y_train == 0]
scaler = StandardScaler()
normal = scaler.fit_transform(normal)
x_test = scaler.transform(x_test)
x_train, x_val = train_test_split(normal, test_size=0.1, random_state=42)

# LSTM needs 3D input: samples, time steps, features.
# Here each row's feature values are treated as a short sequence.
x_train = x_train.reshape(-1, x_train.shape[1], 1)
x_val = x_val.reshape(-1, x_val.shape[1], 1)
x_test = x_test.reshape(-1, x_test.shape[1], 1)


In [3]:
# 2. Create the LSTM encoder and decoder.
encoder = models.Sequential([
    layers.Input(shape=(x_train.shape[1], 1)),
    layers.LSTM(16)
])
decoder = models.Sequential([
    layers.Input(shape=(16,)),
    layers.RepeatVector(x_train.shape[1]),
    layers.LSTM(16, return_sequences=True),
    layers.TimeDistributed(layers.Dense(1))
])
model = models.Sequential([encoder, decoder])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])


In [4]:
# 3. Train the LSTM autoencoder only on normal data.
model.fit(x_train, x_train, validation_data=(x_val, x_val),
          epochs=20, batch_size=512, verbose=2)


Epoch 1/20
7/7 - 18s - 3s/step - loss: 0.9521 - mae: 0.7824 - val_loss: 0.9380 - val_mae: 0.7711
Epoch 2/20
7/7 - 1s - 77ms/step - loss: 0.9291 - mae: 0.7727 - val_loss: 0.9151 - val_mae: 0.7611
Epoch 3/20
7/7 - 0s - 70ms/step - loss: 0.9037 - mae: 0.7617 - val_loss: 0.8892 - val_mae: 0.7498
Epoch 4/20
7/7 - 1s - 73ms/step - loss: 0.8749 - mae: 0.7490 - val_loss: 0.8609 - val_mae: 0.7369
Epoch 5/20
7/7 - 1s - 104ms/step - loss: 0.8453 - mae: 0.7353 - val_loss: 0.8332 - val_mae: 0.7233
Epoch 6/20
7/7 - 1s - 81ms/step - loss: 0.8197 - mae: 0.7236 - val_loss: 0.8142 - val_mae: 0.7138
Epoch 7/20
7/7 - 1s - 79ms/step - loss: 0.8063 - mae: 0.7177 - val_loss: 0.8083 - val_mae: 0.7106
Epoch 8/20
7/7 - 1s - 72ms/step - loss: 0.8034 - mae: 0.7163 - val_loss: 0.8066 - val_mae: 0.7094
Epoch 9/20
7/7 - 1s - 73ms/step - loss: 0.8002 - mae: 0.7143 - val_loss: 0.8037 - val_mae: 0.7077
Epoch 10/20
7/7 - 1s - 79ms/step - loss: 0.7969 - mae: 0.7121 - val_loss: 0.8022 - val_mae: 0.7070
Epoch 11/20
7/7 - 1

In [5]:
# 4. Calculate reconstruction error and detect anomalies.
threshold = np.percentile(np.mean((x_val - model.predict(x_val, verbose=0)) ** 2, axis=(1, 2)), 99)
error = np.mean((x_test - model.predict(x_test, verbose=0)) ** 2, axis=(1, 2))
prediction = (error > threshold).astype(int)
print("Threshold:", threshold)
print(classification_report(y_test, prediction, target_names=["Normal", "Anomaly"], zero_division=0))
print("Number of anomalies detected:", prediction.sum())


Threshold: 2.8605174617958453
              precision    recall  f1-score   support

      Normal       0.96      1.00      0.98       950
     Anomaly       1.00      0.12      0.21        50

    accuracy                           0.96      1000
   macro avg       0.98      0.56      0.60      1000
weighted avg       0.96      0.96      0.94      1000

Number of anomalies detected: 6
